In [0]:
from delta.tables import DeltaTable

In [0]:
tables = {
    "biotech_funding": "deal_id",
    "clinical_trials": "trial_id",
    "disease_burden": ["year", "region", "disease"],
    "drug_approvals": "approval_id",
    "pharma_companies": ["year", "company_name"]
}

In [0]:
 for table, key in tables.items():

    print("=" * 80)
    print(f"MERGING : {table}_silver")

    updates_df = spark.table(f"workspace.default.{table}_silver")

    delta_table = DeltaTable.forName(
        spark,
        f"workspace.default.{table}_silver"
    )

    # Build merge condition
    if isinstance(key, list):
        condition = " AND ".join(
            [f"target.{c}=source.{c}" for c in key]
        )
    else:
        condition = f"target.{key}=source.{key}"

    (
        delta_table.alias("target")
        .merge(
            updates_df.alias("source"),
            condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Merge Completed")

MERGING : biotech_funding_silver
Merge Completed
MERGING : clinical_trials_silver
Merge Completed
MERGING : disease_burden_silver
Merge Completed
MERGING : drug_approvals_silver
Merge Completed
MERGING : pharma_companies_silver
Merge Completed


In [0]:
spark.table("workspace.default.clinical_trials_silver").columns


['trial_id',
 'completion_date',
 'year',
 'sponsor',
 'therapy_area',
 'phase',
 'enrollment_n',
 'duration_months',
 'outcome',
 'is_success',
 'is_failure',
 'estimated_stock_impact_pct',
 'source_file',
 'load_timestamp']

In [0]:
spark.table("workspace.default.drug_approvals_silver").columns


['approval_id',
 'approval_date',
 'year',
 'drug_name',
 'sponsor_company',
 'drug_type',
 'therapy_area',
 'peak_sales_usd_bn_est',
 'is_blockbuster',
 'is_mega_blockbuster',
 'description',
 'is_real_headline',
 'source_file',
 'load_timestamp']

In [0]:
spark.table("workspace.default.pharma_companies_silver").columns

['year',
 'company_name',
 'ticker',
 'country_iso3',
 'segment',
 'revenue_usd_bn',
 'operating_margin_pct',
 'operating_income_usd_bn',
 'rd_spend_usd_bn',
 'pipeline_size_est',
 'source_file',
 'load_timestamp']

In [0]:
for table in tables.keys():

    count = spark.table(
        f"workspace.default.{table}_silver"
    ).count()

    print(f"{table}_silver : {count}")

biotech_funding_silver : 1208
clinical_trials_silver : 599
disease_burden_silver : 3310
drug_approvals_silver : 722
pharma_companies_silver : 489
